# 01 — Training experiments

## Fruit detection with YOLO11

**Purpose:** Train a YOLO11 model to detect apples in orchard images.  
**Workflow:** Short experiments here on Colab → full training run on Kaggle.

# Section 0 - Environment & dependencies

## Libraries

In [1]:
# Install / verify dependencies
!pip install ultralytics supervision --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.9 MB/s eta 0:00:00


In [2]:
import ultralytics

ultralytics.checks()    # prints GPU info, verifies CUDA

Ultralytics 8.4.65 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 20.4/107.7 GB disk)


In [ ]:
# This cell is safe to re-run; pip will skip already-installed packages.
#import subprocess, sys

#subprocess.run([
#    sys.executable,
#    "-m",
#    "pip",
#    "install",
#    "ultralytics>=8.3",
#    "supervision>=0.21",
#    "--quiet"])

In [ ]:
import os, shutil, random, json, yaml, math
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display, Image as IPImage
from ultralytics import YOLO
import torch

## Environment detection

In [15]:
PLATFORM = "GColab"

if PLATFORM == "GColab":
    ROOT_IN = Path("/content/project_fruit_detection/")
    ROOT_OUT = Path("/content/trained_model")
elif PLATFORM == "Kagle":
    ROOT_IN = Path("/kaggle/input/datasets/matarauj/project-fruit-detection/")
    ROOT_OUT = Path("/kaggle/working/")
else:
    ROOT = Path(".").resolve()

In [16]:
####################
# INPUT
####################
DATA_DIR = ROOT_IN / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
ANNOTATIONS_DIR = DATA_DIR / "annotations"
SPLITS_DIR = DATA_DIR / "splits"

DATASET_YAML = SPLITS_DIR / "dataset.yaml"

In [17]:
assert DATASET_YAML.exists(), f'Not found: {DATASET_YAML}'

In [ ]:
print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}  —  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

# Section 1 - Configuration

All training hyperparameters live here. Adjust before each run — no need to hunt through later cells.

| Parameter    | Colab (experiment) | Kaggle (full run) |
|-----------   |------------------- |-------------------|
| `epochs`     | 30–50              | 100               |
| `model_size` | `n` (nano)         | `m` (medium)      |
| `batch`      | 8–16               | 16–32             |

In [1]:
CFG = {
    # ── Dataset ───────────────────────────────────────────────────────────────
    "kaggle_dataset"  : "matarauj/project-fruit-detection",  # Kaggle dataset slug
    "num_classes"     : 1,
    "class_names"     : ["apple"],
    "val_ratio"       : 0.15,
    "test_ratio"      : 0.10,
    "split_seed"      : 42,

    # ── Model ────────────────────────────────────────────────────────────────
    # Sizes: n (nano) → s → m → l → x (extra-large)
    "model_size"      : "n",

    # ── Training ─────────────────────────────────────────────────────────────
    "epochs"          : 50,
    "batch"           : 16,          # Reduce to 8 if you hit GPU OOM
    "imgsz"           : 640,
    "patience"        : 10,          # Early stopping patience
    "lr0"             : 0.01,        # Initial learning rate
    "lrf"             : 0.01,        # Final LR as a fraction of lr0 (cosine schedule)
    "warmup_epochs"   : 3,
    "optimizer"       : "SGD",       # SGD or AdamW
    "weight_decay"    : 0.0005,

    # ── Augmentation ─────────────────────────────────────────────────────────
    "mosaic"          : 1.0,         # Mosaic 4-image composition
    "mixup"           : 0.1,         # MixUp blending
    "fliplr"          : 0.5,         # Horizontal flip probability
    "flipud"          : 0.0,
    "hsv_h"           : 0.015,       # Hue shift
    "hsv_s"           : 0.7,         # Saturation shift
    "hsv_v"           : 0.4,         # Brightness / value shift
    "degrees"         : 5.0,         # Rotation ±degrees
    "translate"       : 0.1,
    "scale"           : 0.5,

    # ── Output ───────────────────────────────────────────────────────────────
    "project"         : "runs",
    "run_name"        : "fruit_v1",
    "save_period"     : 10,          # Save checkpoint every N epochs
    "workers"         : 2,           # DataLoader workers (Colab: 2; Kaggle: 4)
    "seed"            : 42,
    "verbose"         : True
}

print("Configuration loaded:")
for k, v in CFG.items():
    print(f"  {k:<20} {v}")

Configuration loaded:
  kaggle_dataset       matarauj/project-fruit-detection
  num_classes          1
  class_names          ['apple']
  val_ratio            0.15
  test_ratio           0.1
  split_seed           42
  model_size           n
  epochs               50
  batch                16
  imgsz                640
  patience             10
  lr0                  0.01
  lrf                  0.01
  warmup_epochs        3
  optimizer            SGD
  weight_decay         0.0005
  mosaic               1.0
  mixup                0.1
  fliplr               0.5
  flipud               0.0
  hsv_h                0.015
  hsv_s                0.7
  hsv_v                0.4
  degrees              5.0
  translate            0.1
  scale                0.5
  project              runs
  run_name             fruit_v1
  save_period          10
  workers              2
  seed                 42
  verbose              True


# Section 2 - Dataset access

## Mount dataset

Connect the notebook to Drive so that the Kaggle API can be copied. Then, the notebook connects to Kaggle and copy the dataset. Finally, the file is unzipped in the Colab's file space.

In [3]:
from google.colab import drive
import os

In [4]:
!kaggle --version

Kaggle CLI 2.0.2


In [5]:
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
!mkdir ~/.kaggle

In [7]:
!cp '/content/drive/MyDrive/Work/Kaggle_Colab_API/kaggle_colab.json' ~/.kaggle/kaggle.json

In [8]:
!chmod 600 ~/.kaggle/kaggle.json

In [9]:
!kaggle datasets download matarauj/project-fruit-detection

Dataset URL: https://www.kaggle.com/datasets/matarauj/project-fruit-detection
License(s): unknown
100% 232M/232M [00:01<00:00, 145MB/s]



In [10]:
data_root = "/content/project_fruit_detection"
os.makedirs(data_root, exist_ok=True)

os.makedirs('/content/trained_model', exist_ok=True)

In [12]:
!unzip -q project-fruit-detection.zip -d {data_root}